# TabDPT Classifier — DIMER-ready smoke tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)
[![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference)
[![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

This notebook verifies the pinned TabDPT v1.2 integration on a small classification task. Public sample metrics are sanity checks only because TabDPT was pretrained on real-world tables and benchmark overlap cannot be ruled out.


In [ ]:
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git /content/tabdpt-classifier-pipeline
%pip install -q '/content/tabdpt-classifier-pipeline[model]'


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from tabdpt_classifier_pipeline import TabDPTClassificationPipeline

frame = load_breast_cancer(as_frame=True).frame
train, test = train_test_split(frame, test_size=0.2, random_state=42, stratify=frame['target'])
print(train.shape, test.shape)


In [ ]:
# Explicitly disable FlashAttention for portability to Colab/Kaggle Tesla T4 (sm_75) GPUs.
pipe = TabDPTClassificationPipeline(compile_model=False, use_flash=False)
pipe.fit(train, target_column='target')
print('classes:', pipe.class_labels_)


In [ ]:
metrics = pipe.evaluate(test, n_ensembles=2, context_size=512, batch_size=512, seed=42)
metrics


## Production note
For DIMER deployment, mount/cache the verified `tabdpt1_2.safetensors` artifact instead of relying on an internet download. The pipeline auto-enables FlashAttention only on compatible CUDA devices (compute capability 8.0+); callers may still override `use_flash` explicitly. Preserve application-specific train/validation/test splits for leakage-sensitive data.
